# Branch 2 — Feature Extraction & Quick Baseline

Test feature extraction approaches (TF-IDF + numerical features)
for Anomaly Detection (Isolation Forest / One-Class SVM).

**Pipeline:**
1. Load raw CSV
2. Canonicalize
3. Extract features (TF-IDF char n-gram + numerical)
4. Train Isolation Forest on normal only
5. Evaluate on anomalous

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in path
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score

from src.utils import load_config
from src.preprocessing.canonicalize import canonicalize_request
from src.models.nhanh2_embed import extract_features

In [ ]:
cfg = load_config()
data_path = Path(cfg.get_path("paths.data_raw")) / "d3_csic2010_raw.csv"
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows")
df.head(2)

## 1. Canonicalize all requests

In [ ]:
canon = []
for _, row in df.iterrows():
    canon.append(canonicalize_request(
        row["method"], row["path"], row["query"], row["body"], cfg
    ))
df_canon = pd.DataFrame(canon)
df_canon["label"] = df["label"].values
print(f"Canonicalized {len(df_canon)} requests")
df_canon.head(2)

## 2. Extract features

In [ ]:
X, vec = extract_features(df_canon, cfg=cfg, fit=True)
y = (df_canon["label"] == "anomalous").astype(int).values
print(f"Feature matrix shape: {X.shape}")

## 3. Split: train on normal only, test on all

In [ ]:
normal_idx = df_canon["label"] == "normal"
X_train = X[normal_idx]
X_test = X[~normal_idx]
y_train = np.zeros(X_train.shape[0])
y_test = y[~normal_idx]
print(f"Train (normal only): {X_train.shape[0]} rows")
print(f"Test (anomalous): {X_test.shape[0]} rows")

## 4. Train Isolation Forest

In [ ]:
model = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train)
print("Model trained")

## 5. Evaluate

In [ ]:
y_pred = model.predict(X_test)
# Isolation Forest: 1 = inlier (normal), -1 = outlier (anomalous)
y_pred_bin = (y_pred == -1).astype(int)

print(classification_report(y_test, y_pred_bin, target_names=["normal", "anomalous"]))
print(f"ROC-AUC: {roc_auc_score(y_test, -model.score_samples(X_test)):.4f}")

## 6. Quick comparison: TF-IDF only vs combined

In [ ]:
tfidf_only = X[:, :-9]  # all but last 9 numeric cols
md2 = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=-1)
md2.fit(tfidf_only[normal_idx])
y_pred2 = md2.predict(tfidf_only[~normal_idx])
y_pred_bin2 = (y_pred2 == -1).astype(int)
print("TF-IDF only:")
print(classification_report(y_test, y_pred_bin2, target_names=["normal", "anomalous"]))
print(f"ROC-AUC: {roc_auc_score(y_test, -md2.score_samples(tfidf_only[~normal_idx])):.4f}")

### Nhận xét

- So sánh kết quả TF-IDF only vs combined (TF-IDF + numerical) để chọn feature set cuối.
- Nếu cả 2 thấp, có thể thử One-Class SVM hoặc tuning contamination.
- Ghi lại kết quả để báo cáo.